# momentum-buffer-update — faded example 1: Apply momentum buffer update to a two-parameter group

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `momentum-buffer-update`. Running the beacon reports progress on the `Optimizer: Momentum buffer` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Momentum buffer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`momentum-buffer-update`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "momentum-buffer-update"
DD_SUBTOPIC = "Optimizer: Momentum buffer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The momentum recurrence `b ← μ·b + g` must be applied independently to each parameter's buffer. A typical optimizer iterates over all parameters, looks up (or initializes) each parameter's buffer, and applies the in-place update. The `.copy_()` call ensures the existing buffer tensor is mutated rather than replaced, preserving the accumulated state.

## Faded exercise 1

Implement `momentum_step(params, grads, buffers, mu)` that applies one step of `b ← μ·b + g` for each (param, grad, buffer) triple.

For each position `i`:
1. Update `buffers[i]` **in-place**: `buffers[i].copy_(mu * buffers[i] + grads[i])`.
2. Append `buffers[i]` to the output list (the effective gradient).

Return the list of effective gradients.

Your task: **fill in the in-place buffer update and the append**.

**Fill in:** The in-place buffer update b.copy_(mu * b + g) and appending the updated buffer to g_eff_list.

In [ ]:
import torch

def momentum_step(params, grads, buffers, mu):
    g_eff_list = []
    for b, g in zip(buffers, grads):
        b.copy_(mu * b + g)
        g_eff_list.append(b)
    return g_eff_list

def _test():
    import torch
    params = [torch.randn(3), torch.randn(2)]
    grads  = [torch.tensor([1.0, 0.5, -1.0]), torch.tensor([2.0, -0.5])]
    bufs   = [torch.zeros(3), torch.zeros(2)]
    mu = 0.9
    out1 = momentum_step(params, grads, bufs, mu)
    assert torch.allclose(bufs[0], grads[0])
    out2 = momentum_step(params, grads, bufs, mu)
    expected0 = 0.9 * grads[0] + grads[0]
    assert torch.allclose(bufs[0], expected0, atol=1e-5), f"got {bufs[0]}, expected {expected0}"


def _test():
    import torch
    params = [torch.randn(3), torch.randn(2)]
    grads  = [torch.tensor([1.0, 0.5, -1.0]), torch.tensor([2.0, -0.5])]
    bufs   = [torch.zeros(3), torch.zeros(2)]
    mu = 0.9
    out1 = momentum_step(params, grads, bufs, mu)
    # After step 1 buffers should equal grads (since buf was zero)
    assert torch.allclose(bufs[0], grads[0]), f"step1 buf[0] wrong: {bufs[0]}"
    assert torch.allclose(bufs[1], grads[1]), f"step1 buf[1] wrong: {bufs[1]}"
    out2 = momentum_step(params, grads, bufs, mu)
    # After step 2: buf = 0.9*grad + grad = 1.9*grad
    expected0 = 0.9 * grads[0] + grads[0]
    expected1 = 0.9 * grads[1] + grads[1]
    assert torch.allclose(bufs[0], expected0, atol=1e-5), f"step2 buf[0]: got {bufs[0]}, expected {expected0}"
    assert torch.allclose(bufs[1], expected1, atol=1e-5)
    # Returned g_eff should be the same objects as the buffers
    assert out2[0] is bufs[0]
    assert out2[1] is bufs[1]


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch

def momentum_step(params, grads, buffers, mu):
    g_eff_list = []
    for b, g in zip(buffers, grads):
        b.copy_(mu * b + g)
        g_eff_list.append(b)
    return g_eff_list

def _test():
    import torch
    params = [torch.randn(3), torch.randn(2)]
    grads  = [torch.tensor([1.0, 0.5, -1.0]), torch.tensor([2.0, -0.5])]
    bufs   = [torch.zeros(3), torch.zeros(2)]
    mu = 0.9
    out1 = momentum_step(params, grads, bufs, mu)
    assert torch.allclose(bufs[0], grads[0])
    out2 = momentum_step(params, grads, bufs, mu)
    expected0 = 0.9 * grads[0] + grads[0]
    assert torch.allclose(bufs[0], expected0, atol=1e-5), f"got {bufs[0]}, expected {expected0}"
```
</details>